# FedLR Re-Run — Ceftriaxone x *E. coli*

**Re-run only FedLR with per-site tuned C, then update the last run's results in-place.**

Set `TARGET_RUN` below, then run all cells.

| What's reloaded | What's re-run |
|---|---|
| Data + preprocessing | Per-site LR grid (Option B: worst-site C) |
| Centralized MLP/RF, FedAvg, FedProx, FedRF | FedLR with correct C |
| All existing per-round CSVs | Convergence plot + heatmaps |
| Best MLP/RF params | results/fedlr_per_round.csv + final_results.csv |

The target run's existing files are overwritten with updated results.

In [ ]:
# ── CONFIG ──
TARGET_RUN = "Analysis"   # <-- which run to update
NUM_ROUNDS = 30

In [ ]:
!pip install "flwr[simulation]" maldideepkit seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
import warnings, os, io, shutil, json as _json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (train_test_split, GridSearchCV, StratifiedKFold)
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score)

from maldideepkit.base.data import fit_input_transform, apply_input_transform

import flwr as fl

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
np.random.seed(SEED)
FL_DEVICE = "cpu"
C_GRID = np.linspace(5e-5, 1e-3, 15)
THRESHOLDS = np.linspace(0.05, 0.95, 91)
print(f"Flower: {fl.__version__}  |  Target: {TARGET_RUN}")

In [ ]:
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

DRUG_NAME = "Ceftriaxone"
SPECIES = "Escherichia coli"

PROJECT_DIR = DRYAD / "Processed/Processing/Analysis/06-Ceftazidime-E-coli"
RUN_DIR = PROJECT_DIR / TARGET_RUN
OUT_DIR = RUN_DIR / "results"
MODEL_DIR = RUN_DIR / "models"
OUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / DRUG_NAME / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / DRUG_NAME / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / DRUG_NAME / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / DRUG_NAME / "data.csv",
}
SITE_ORDER = ["A", "B", "C", "D"]
print(f"Run dir: {RUN_DIR}")
print(f"Drug: {DRUG_NAME}  |  Species: {SPECIES}")

In [ ]:
# ── Load Ceftriaxone + E. coli ──
raw_data = {}
for site, path in SITES_PATHS.items():
    df = pd.read_csv(path)
    df_eco = df[df["species"] == SPECIES].copy()
    bin_cols = [c for c in df_eco.columns if c.startswith("bin_")]
    X = df_eco[bin_cols].to_numpy(dtype="float32")
    y = df_eco["label"].to_numpy(dtype="int64")
    raw_data[site] = (X, y)
    nr, ns = (y==1).sum(), (y==0).sum()
    print(f"  Site {site}: {len(y)} samples ({ns} S, {nr} R, {nr/len(y)*100:.1f}% R)")
print(f"Total: {sum(len(raw_data[s][1]) for s in SITE_ORDER)}")

In [ ]:
# ── Per-site 90/10 label-stratified split ──
client_train, client_test = {}, {}
site_seeds = {"A": 42, "B": 123, "C": 456, "D": 789}
for site in SITE_ORDER:
    X, y = raw_data[site]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.10, stratify=y, random_state=site_seeds[site])
    client_train[site] = (X_tr, y_tr)
    client_test[site] = (X_te, y_te)
    print(f"  Site {site}: train={len(X_tr)} test={len(X_te)}")

In [ ]:
# ── Per-site preprocessing ──
client_train_pp, client_test_pp = {}, {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train[site]
    X_te, y_te = client_test[site]
    state = fit_input_transform(X_tr, "log1p+standardize")
    client_train_pp[site] = (apply_input_transform(X_tr, state), y_tr)
    client_test_pp[site] = (apply_input_transform(X_te, state), y_te)
print("Per-site preprocessing done.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Option B: Per-site LR grid search -> worst-site optimal C
# ═══════════════════════════════════════════════════════════════════════════

print("\n=== LR Grid Search (Option B, per-site) ===")
site_lr_scores = {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train_pp[site]
    grid = GridSearchCV(
        LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced",
                           max_iter=5000, random_state=SEED),
        param_grid={"C": C_GRID}, cv=3, scoring="balanced_accuracy", n_jobs=-1)
    grid.fit(X_tr, y_tr)
    for i, params in enumerate(grid.cv_results_["params"]):
        key = float(params["C"])
        score = grid.cv_results_["mean_test_score"][i]
        site_lr_scores.setdefault(key, []).append(score)
    print(f"  Site {site}: best C={grid.best_params_['C']:.2e}, BA={grid.best_score_:.4f}")

# Worst-site optimization
BEST_LR_C = max(site_lr_scores, key=lambda c: min(site_lr_scores[c]))
print(f"\nBest LR C (worst-site): {BEST_LR_C:.2e}")

# Per-site threshold tuning (worst-site optimal)
lr_thresh_scores = {t: [] for t in THRESHOLDS}
for site in SITE_ORDER:
    X_tr, y_tr = client_train_pp[site]
    from sklearn.model_selection import cross_val_predict
    cv_proba = cross_val_predict(
        LogisticRegression(C=BEST_LR_C, penalty="l2", solver="lbfgs",
                           class_weight="balanced", max_iter=5000, random_state=SEED),
        X_tr, y_tr, cv=3, method="predict_proba", n_jobs=-1)[:, 1]
    for t in THRESHOLDS:
        lr_thresh_scores[t].append(balanced_accuracy_score(y_tr, cv_proba >= t))

BEST_LR_THRESH = max(THRESHOLDS, key=lambda t: min(lr_thresh_scores[t]))
print(f"LR threshold (worst-site): {BEST_LR_THRESH:.3f}")

In [ ]:
# ── Load existing run's params + per-round CSVs ──
if (OUT_DIR / "best_params_used.txt").exists():
    params = {}
    with open(OUT_DIR / "best_params_used.txt") as f:
        for line in f:
            if "=" in line:
                k, v = line.strip().split("=", 1)
                params[k] = v
    BEST_MLP_LR = float(params["BEST_MLP_LR"])
    BEST_MLP_DH = float(params["BEST_MLP_DH"])
    BEST_MLP_THRESH = float(params["BEST_MLP_THRESH"])
    BEST_RF_THRESH = float(params["BEST_RF_THRESH"])
    RF_PARAMS = eval(params["RF_PARAMS"])
    print(f"Loaded MLP: lr={BEST_MLP_LR:.2e} drop={BEST_MLP_DH:.1f} thresh={BEST_MLP_THRESH:.3f}")
    print(f"Loaded RF:  {RF_PARAMS}")
else:
    raise FileNotFoundError("best_params_used.txt not found in run directory")

fedavg_hist = []; fedlr_hist = []; fedrf_hist = []; fedprox_histories = {}
if (OUT_DIR / "fedavg_per_round.csv").exists():
    fedavg_hist = pd.read_csv(OUT_DIR / "fedavg_per_round.csv").to_dict("records")
    print(f"Loaded fedavg: {len(fedavg_hist)} rounds")
if (OUT_DIR / "fedrf_per_round.csv").exists():
    fedrf_hist = pd.read_csv(OUT_DIR / "fedrf_per_round.csv").to_dict("records")
    print(f"Loaded fedrf: {len(fedrf_hist)} rounds")
for f in sorted(OUT_DIR.glob("fedprox_mu*_per_round.csv")):
    mu = float(f.stem.replace("fedprox_mu","").replace("_per_round",""))
    fedprox_histories[mu] = pd.read_csv(f).to_dict("records")
    print(f"Loaded fedprox mu={mu}: {len(fedprox_histories[mu])} rounds")

# Load centralized + cross-site from existing final_results.csv
centralized_mlp_results = {}; centralized_rf_results = {}
cross_site_results = {}; cross_site_rf_results = {}
if (OUT_DIR / "final_results.csv").exists():
    df_old = pd.read_csv(OUT_DIR / "final_results.csv")
    for meth, target in [("Centralized MLP", centralized_mlp_results),
                          ("Centralized RF", centralized_rf_results),
                          ("Cross-Site MLP", cross_site_results),
                          ("Cross-Site RF", cross_site_rf_results)]:
        row = df_old[df_old["Method"] == meth]
        if len(row) > 0:
            r = row.iloc[0]
            for site in SITE_ORDER:
                col = f"{site}_BalAcc"
                if col in r and not pd.isna(r[col]):
                    target[f"{site}_BalAcc"] = float(r[col])
                col_auc = f"{site}_AUC"
                if col_auc in r and not pd.isna(r[col_auc]):
                    target[f"{site}_AUC"] = float(r[col_auc])
            if "All_BalAcc" in r and not pd.isna(r["All_BalAcc"]):
                target["All_BalAcc"] = float(r["All_BalAcc"])
            if "All_AUC" in r and not pd.isna(r["All_AUC"]):
                target["All_AUC"] = float(r["All_AUC"])
    print("Loaded centralized + cross-site from final_results.csv")

In [ ]:
# ── FedLR Client with correct C ──
class FedLRClient(fl.client.NumPyClient):
    def __init__(self, cid, X_train, y_train):
        self.cid = cid; self.X_train, self.y_train = X_train, y_train
        nf = X_train.shape[1]
        self.model = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="saga", max_iter=1,
                                        warm_start=True, class_weight="balanced", random_state=SEED)
        self.model.classes_ = np.array([0,1]); self.model.coef_ = np.zeros((1,nf)); self.model.intercept_ = np.zeros(1)
    def get_parameters(self, config): return [self.model.coef_.ravel(), self.model.intercept_]
    def set_parameters(self, params):
        self.model.coef_ = params[0].reshape(1,-1); self.model.intercept_ = params[1]
    def fit(self, parameters, config):
        self.set_parameters(parameters)
        with warnings.catch_warnings(): warnings.simplefilter("ignore"); self.model.fit(self.X_train, self.y_train)
        return (self.get_parameters({}), len(self.X_train), {"num_examples": len(self.X_train)})

In [ ]:
# ── LR-specific eval + checkpoint ──
def get_lr_eval_fn(test_dict, threshold, hist_list):
    def evaluate(server_round, parameters, config):
        lr = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="lbfgs",
                                class_weight="balanced", max_iter=5000)
        lr.classes_ = np.array([0,1])
        lr.coef_ = parameters[0].reshape(1,-1); lr.intercept_ = parameters[1]
        record = {"round": server_round}
        ap, al = [], []
        for site in SITE_ORDER:
            X_tt, y_tt = test_dict[site]
            proba = lr.predict_proba(X_tt)[:, 1]
            preds = proba >= threshold
            record[f"{site}_BalAcc"] = float(balanced_accuracy_score(y_tt, preds))
            record[f"{site}_AUC"] = float(roc_auc_score(y_tt, proba))
            ap.append(proba); al.append(y_tt)
        apc = np.concatenate(ap); alc = np.concatenate(al)
        record["All_BalAcc"] = float(balanced_accuracy_score(alc, apc >= threshold))
        record["All_AUC"] = float(roc_auc_score(alc, apc))
        hist_list.append(record)
        return (1.0 - record["All_BalAcc"], record)
    return evaluate

class CheckpointLRFedAvg(fl.server.strategy.FedAvg):
    def __init__(self, model_dir, **kwargs):
        super().__init__(**kwargs)
        self.model_dir = Path(model_dir) / "fedavg_lr"
        self.model_dir.mkdir(parents=True, exist_ok=True)
    def aggregate_fit(self, server_round, results, failures):
        round_dir = self.model_dir / f"round_{server_round:03d}"
        round_dir.mkdir(parents=True, exist_ok=True)
        for cp, fit_res in results:
            ndarrays = fl.common.parameters_to_ndarrays(fit_res.parameters)
            np.savez(round_dir / f"client_{cp.cid}.npz", coef=ndarrays[0], intercept=ndarrays[1])
        aggregated, metrics = super().aggregate_fit(server_round, results, failures)
        if aggregated is not None:
            nd = fl.common.parameters_to_ndarrays(aggregated)
            np.savez(round_dir / "global_model.npz", coef=nd[0], intercept=nd[1])
        return aggregated, metrics

In [ ]:
def lr_client_fn(cid):
    site = SITE_ORDER[int(cid)]
    return FedLRClient(cid, *client_train_pp[site]).to_client()

In [ ]:
# ── Run FedLR with correct C ──
eval_hist_fedlr = []
_eval_fn = get_lr_eval_fn(client_test_pp, BEST_LR_THRESH, eval_hist_fedlr)

n_feat = client_train_pp[SITE_ORDER[0]][0].shape[1]
lr_init = LogisticRegression(C=BEST_LR_C, penalty="l2", solver="saga",
                              max_iter=1, warm_start=True, class_weight="balanced", random_state=SEED)
lr_init.classes_ = np.array([0,1]); lr_init.coef_ = np.zeros((1,n_feat)); lr_init.intercept_ = np.zeros(1)

strategy = CheckpointLRFedAvg(MODEL_DIR,
    fraction_fit=1.0, fraction_evaluate=0.0,
    min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
    evaluate_fn=_eval_fn,
    initial_parameters=fl.common.ndarrays_to_parameters([lr_init.coef_.ravel(), lr_init.intercept_]))
strategy.model_dir = Path(MODEL_DIR) / "fedavg_lr"

print(f"\n=== FedAvg LR (C={BEST_LR_C:.2e}) ===")
fl.simulation.start_simulation(
    client_fn=lr_client_fn, num_clients=4,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy, client_resources={"num_cpus": 1, "num_gpus": 0})

fedlr_hist = eval_hist_fedlr.copy()
print(f"FedLR done. {len(fedlr_hist)} rounds.")
last = fedlr_hist[-1] if fedlr_hist else {}
print(f"  Final All_BalAcc: {last.get('All_BalAcc', np.nan):.4f}")

In [ ]:
# ── Assemble final results DataFrame ──
def best_metrics(h):
    if not h: return {}, 0
    skip0 = h[1:]
    best = max(skip0, key=lambda r: r.get("All_BalAcc", 0))
    return best, int(best.get("round", 0))

rows = []
def add_row(method, cs_source=None, fed_hist=None):
    r = {"Method": method}; peak_round = ""
    for site in SITE_ORDER:
        if cs_source:
            r[f"{site}_BalAcc"] = cs_source.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = cs_source.get(f"{site}_AUC", np.nan)
        elif fed_hist is not None:
            lm, pr = best_metrics(fed_hist)
            r[f"{site}_BalAcc"] = lm.get(f"{site}_BalAcc", np.nan)
            r[f"{site}_AUC"] = lm.get(f"{site}_AUC", np.nan)
            peak_round = f" (r{pr})"
    if fed_hist is not None:
        lm, pr = best_metrics(fed_hist)
        r["All_BalAcc"] = lm.get("All_BalAcc", np.nan)
        r["All_AUC"] = lm.get("All_AUC", np.nan)
        r["Peak_Round"] = int(pr)
    elif cs_source:
        r["All_BalAcc"] = cs_source.get("All_BalAcc", np.nan)
        r["All_AUC"] = cs_source.get("All_AUC", np.nan)
        r["Peak_Round"] = 0
    r["Label"] = method + peak_round
    rows.append(r)

add_row("Centralized MLP", cs_source=centralized_mlp_results)
add_row("Centralized RF", cs_source=centralized_rf_results)
if fedavg_hist: add_row("FL FedAvg (MLP)", fed_hist=fedavg_hist)
for mu in sorted(fedprox_histories.keys()):
    add_row(f"FL FedProx mu={mu} (MLP)", fed_hist=fedprox_histories[mu])
add_row("FL FedAvg (LR)", fed_hist=fedlr_hist)
if fedrf_hist: add_row("FL FedRF (Trees)", fed_hist=fedrf_hist)
if cross_site_results:
    r = {"Method": "Cross-Site MLP", "Label": "Cross-Site MLP", "Peak_Round": 0}
    for site in SITE_ORDER:
        r[f"{site}_BalAcc"] = cross_site_results.get(f"{site}_BalAcc", np.nan)
        r[f"{site}_AUC"] = cross_site_results.get(f"{site}_AUC", np.nan)
    r["All_BalAcc"] = cross_site_results.get("All_BalAcc", np.nan)
    r["All_AUC"] = cross_site_results.get("All_AUC", np.nan)
    rows.append(r)
if cross_site_rf_results:
    r = {"Method": "Cross-Site RF", "Label": "Cross-Site RF", "Peak_Round": 0}
    for site in SITE_ORDER:
        r[f"{site}_BalAcc"] = cross_site_rf_results.get(f"{site}_BalAcc", np.nan)
        r[f"{site}_AUC"] = cross_site_rf_results.get(f"{site}_AUC", np.nan)
    r["All_BalAcc"] = cross_site_rf_results.get("All_BalAcc", np.nan)
    r["All_AUC"] = cross_site_rf_results.get("All_AUC", np.nan)
    rows.append(r)

df_results = pd.DataFrame(rows)
cols = ["Method", "Label", "Peak_Round"] + [f"{s}_BalAcc" for s in SITE_ORDER] + ["All_BalAcc"]
print(df_results[cols].to_string(index=False))

In [ ]:
# ── Convergence plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
if fedavg_hist:
    for site in SITE_ORDER:
        vals = [h.get(f"{site}_BalAcc", np.nan) for h in fedavg_hist]
        ax.plot(range(1, len(vals)+1), vals, marker='.', label=f"Site {site}")
    vals_all = [h.get("All_BalAcc", np.nan) for h in fedavg_hist]
    ax.plot(range(1, len(vals_all)+1), vals_all, 'k-', lw=2, label="All")
ax.set_title("FedAvg MLP — Per-Site"); ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)

ax = axes[1]
plots = [("FedAvg MLP", fedavg_hist, "#ff7f0e", "-")]
for mu in sorted(fedprox_histories.keys()):
    plots.append((f"FedProx mu={mu}", fedprox_histories[mu], "#d62728", "--"))
plots.append(("FedAvg LR", fedlr_hist, "#1f77b4", "-."))
if fedrf_hist: plots.append(("FedRF", fedrf_hist, "#2ca02c", "-"))
for label, hist, c, ls in plots:
    vals = [h.get("All_BalAcc", np.nan) for h in hist]
    ax.plot(range(1, len(vals)+1), vals, color=c, ls=ls, lw=2, label=label)
    skip0 = hist[1:]
    if skip0:
        best = max(skip0, key=lambda h: h.get("All_BalAcc", 0))
        pr = int(best.get("round", 0))
        ax.axvline(pr, color=c, ls=':', alpha=0.3, lw=1)
if centralized_mlp_results.get("All_BalAcc"):
    ax.axhline(centralized_mlp_results["All_BalAcc"], color='gray', ls=':', lw=2, label='Centralized MLP')
if centralized_rf_results.get("All_BalAcc"):
    ax.axhline(centralized_rf_results["All_BalAcc"], color='gray', ls='--', lw=2, label='Centralized RF')
ax.set_title(f"All Methods — All-Site BalAcc (LR C={BEST_LR_C:.2e})"); ax.set_xlabel("Round"); ax.set_ylabel("BalAcc")
ax.legend(fontsize=7); ax.grid(True, ls='--', alpha=0.5); ax.set_ylim(0.3, 1.0)

fig.suptitle(f"{DRUG_NAME} x {SPECIES} — FL Convergence", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "convergence.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Heatmap: Balanced Accuracy ──
ba_data = {}
for _, r in df_results.iterrows():
    ba_data[r["Label"]] = {f"Site {s}": r[f"{s}_BalAcc"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_BalAcc", np.nan)):
        ba_data[r["Label"]]["All"] = r["All_BalAcc"]
df_ba_hm = pd.DataFrame(ba_data).T
df_ba_hm = df_ba_hm[[c for c in [f"Site {s}" for s in SITE_ORDER] + ["All"] if c in df_ba_hm.columns]]
fig, ax = plt.subplots(figsize=(10, max(4, len(df_ba_hm)*0.5)))
sns.heatmap(df_ba_hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0,
            linewidths=1.0, linecolor="white", cbar_kws={"label": "Balanced Accuracy"}, ax=ax)
ax.set_title(f"{DRUG_NAME} x {SPECIES} — BalAcc", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "heatmap_balacc.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Heatmap: AUC ──
auc_data = {}
for _, r in df_results.iterrows():
    auc_data[r["Label"]] = {f"Site {s}": r[f"{s}_AUC"] for s in SITE_ORDER}
    if not np.isnan(r.get("All_AUC", np.nan)):
        auc_data[r["Label"]]["All"] = r["All_AUC"]
df_auc_hm = pd.DataFrame(auc_data).T
df_auc_hm = df_auc_hm[[c for c in [f"Site {s}" for s in SITE_ORDER] + ["All"] if c in df_auc_hm.columns]]
fig, ax = plt.subplots(figsize=(10, max(4, len(df_auc_hm)*0.5)))
sns.heatmap(df_auc_hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.4, vmax=1.0,
            linewidths=1.0, linecolor="white", cbar_kws={"label": "AUC-ROC"}, ax=ax)
ax.set_title(f"{DRUG_NAME} x {SPECIES} — AUC-ROC", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "heatmap_auc.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Save results ──
df_results.to_csv(OUT_DIR / "final_results.csv", index=False)
pd.DataFrame(fedlr_hist).to_csv(OUT_DIR / "fedlr_per_round.csv", index=False)
print(f"Overwrote fedlr_per_round.csv + final_results.csv")

# Archive notebook
import shutil
src_nb = Path("06-03b-Re-Run-lr-Ceftriaxone-E-coli-Fed.ipynb")
if not src_nb.exists():
    import glob as _g
    candidates = list(_g.glob("/content/**/06-03b-Re-Run-lr*.ipynb", recursive=True))
    if candidates: src_nb = Path(candidates[0])
if src_nb.exists():
    shutil.copy(str(src_nb), str(OUT_DIR / "rerun_lr_notebook.ipynb"))
    print("Notebook archived.")

print(f"\n{'='*60}")
print(f"  Done. Results updated in {OUT_DIR.resolve()}")
print(f"  LR C = {BEST_LR_C:.2e}  |  threshold = {BEST_LR_THRESH:.3f}")
for f in sorted(OUT_DIR.glob("*")):
    print(f"    {f.name}")

---
**Done.** FedLR re-run complete with per-site tuned C value. Results overwritten in the run directory's `results/` folder.